# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv(r"D:\Internship\Week1\Task1-Week1\data\raw\content_refresh_anonymized.csv")

In [5]:
df.head(50)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.00,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.00,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.00,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.00,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.00,good,page_3_5,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,15000-25000,0.03,8.5,0.00,25.00,0.00,good,page_1,down,-38.9
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,15000-25000,0.00,7.0,0.00,0.00,0.00,low,page_1,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,NaN,0.06,21.2,3.57,7.14,0.00,moderate,page_3_5,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,15000-25000,0.09,46.0,5.88,6.25,0.00,excellent,page_3_5,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,NaN,0.16,4.9,0.00,0.00,0.00,moderate,page_1,down,-29.2


In [6]:
df.shape

(30000, 44)

In [7]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

I prioritize pages that have not been updated for at least 180 days and still receive high impressions. Older pages with strong visibility are more likely to benefit from a content refresh.

### Reason Code

STALE_HIGH_IMPRESSIONS

### Action Label

Refresh Content

In [26]:
# Signal Test 1 - Staleness

df["stale_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0,90,180,365,1000],
    labels=["0-90","91-180","181-365","365+"]
)

signal1 = df.groupby("stale_bucket", observed=False).agg(
    n=("content_id","count"),
    avg_impressions=("impressions_90d","mean")
)

signal1

,n,avg_impressions
stale_bucket,,
0-90,20655,4219.161317
91-180,9171,7486.665140
181-365,169,1206.893491
365+,5,8.200000


### Signal Test 1 Verdict

**Signal:** Days Since Last Update

**Verdict:** MIXED

Pages updated 91–180 days ago have the highest average impressions, while pages older than 180 days receive much lower average impressions. This suggests that staleness alone is not enough. Combining staleness with high impressions is a more reliable baseline rule.

In [27]:
# Signal Test 2 - Impressions

df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0,500,2000,10000,100000],
    labels=["Low","Medium","High","Very High"]
)

signal2 = df.groupby("volume_bucket", observed=False).agg(
    n=("content_id","count"),
    avg_days=("days_since_last_update","mean")
)

signal2

,n,avg_days
volume_bucket,,
Low,13285,37.874219
Medium,6502,48.809136
High,6611,53.298442
Very High,3434,58.208794


### Signal Test 2 Verdict

**Signal:** Impressions

**Verdict:** CONFIRMED

Pages with higher impressions offer greater opportunities for improvement because refreshing highly visible content can produce a larger impact.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# Rule conditions
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

In [9]:
# Transparent baseline score
df["score"] = stale * visible * df["impressions_90d"]

In [10]:
# Reason code
df["reason_code"] = np.where(
    df["score"] > 0,
    "STALE_HIGH_IMPRESSIONS",
    "NO_ACTION"
)

In [11]:
# Action label
df["action"] = np.where(
    df["score"] > 0,
    "Refresh Content",
    "Monitor"
)


In [12]:
# Rank
ranked = df.sort_values("score", ascending=False)

ranked.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code,action
16751,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,5125.0,33705.0,...,0.84,24.11,0.0,excellent,striking,down,-85.6,61678,STALE_HIGH_IMPRESSIONS,Refresh Content
16514,content_7368877ea310,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,2591.0,16498.0,...,3.66,42.99,0.0,excellent,page_3_5,down,-81.5,59472,STALE_HIGH_IMPRESSIONS,Refresh Content
7021,content_1bfaa38ff26c,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3861.0,24672.0,...,3.75,43.33,0.0,good,page_3_5,down,-74.7,25715,STALE_HIGH_IMPRESSIONS,Refresh Content
21268,content_0a91db491d14,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3478.0,21948.0,...,5.13,41.76,0.0,good,striking,down,-51.8,13299,STALE_HIGH_IMPRESSIONS,Refresh Content
11489,content_5feee3994adb,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,transactional,3590.0,22780.0,...,0.00,40.00,0.0,good,page_3_5,down,-89.1,7812,STALE_HIGH_IMPRESSIONS,Refresh Content


In [14]:
import os

In [15]:
os.makedirs("../../work/outputs", exist_ok=True)

ranked.to_csv(
    "../../work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")

CSV written successfully.


Show Top 20

In [21]:
top20 = ranked[
    [
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "score",
        "reason_code",
        "action"
    ]
].head(20)

top20

,content_id,days_since_last_update,impressions_90d,score,reason_code,action
16751,content_cf56e2e2e282,194,61678,61678,STALE_HIGH_IMPRESSIONS,Refresh Content
16514,content_7368877ea310,194,59472,59472,STALE_HIGH_IMPRESSIONS,Refresh Content
7021,content_1bfaa38ff26c,194,25715,25715,STALE_HIGH_IMPRESSIONS,Refresh Content
21268,content_0a91db491d14,193,13299,13299,STALE_HIGH_IMPRESSIONS,Refresh Content
11489,content_5feee3994adb,194,7812,7812,STALE_HIGH_IMPRESSIONS,Refresh Content
12045,content_c2d929d83eaa,193,7558,7558,STALE_HIGH_IMPRESSIONS,Refresh Content
698,content_b16bd7307b39,194,4590,4590,STALE_HIGH_IMPRESSIONS,Refresh Content
5327,content_fe16a55cd13d,194,4556,4556,STALE_HIGH_IMPRESSIONS,Refresh Content
26810,content_ecb6215e79fd,194,4429,4429,STALE_HIGH_IMPRESSIONS,Refresh Content
20837,content_928af3e22c80,193,1697,1697,STALE_HIGH_IMPRESSIONS,Refresh Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

| Rank | Action          | Reason Code            | Confidence | What would make it wrong                                                                           |
| ---- | --------------- | ---------------------- | ---------- | -------------------------------------------------------------------------------------------------- |
| 1    | Refresh Content | STALE_HIGH_IMPRESSIONS | High       | The page may have already been refreshed recently, but the update is not reflected in the dataset. |
| 2    | Refresh Content | STALE_HIGH_IMPRESSIONS | High       | High impressions may be driven by temporary search demand rather than outdated content.            |
| 3    | Refresh Content | STALE_HIGH_IMPRESSIONS | High       | The page may still satisfy user intent even though it is old.                                      |
| 4    | Refresh Content | STALE_HIGH_IMPRESSIONS | High       | The content may already be accurate and require only minor edits.                                  |
| 5    | Refresh Content | STALE_HIGH_IMPRESSIONS | High       | Strong visibility alone does not guarantee that a refresh will improve performance.                |
| 6    | Refresh Content | STALE_HIGH_IMPRESSIONS | High       | The page may already be scheduled for an update by the content team.                               |
| 7    | Refresh Content | STALE_HIGH_IMPRESSIONS | Medium     | High impressions may be caused by seasonal trends instead of content quality.                      |
| 8    | Refresh Content | STALE_HIGH_IMPRESSIONS | Medium     | The page could be evergreen content that remains useful without changes.                           |
| 9    | Refresh Content | STALE_HIGH_IMPRESSIONS | Medium     | Search demand may naturally fluctuate, reducing the expected impact of a refresh.                  |
| 10   | Refresh Content | STALE_HIGH_IMPRESSIONS | Medium     | Manual review may show the content is still factually correct and complete.                        |
| 11   | Refresh Content | STALE_HIGH_IMPRESSIONS | Medium     | The page may require only metadata optimization rather than a full content refresh.                |
| 12   | Refresh Content | STALE_HIGH_IMPRESSIONS | Medium     | A recent update may not yet be reflected in the historical data.                                   |
| 13   | Refresh Content | STALE_HIGH_IMPRESSIONS | Medium     | The page has lower impressions than the top-ranked items, so the business impact may be smaller.   |
| 14   | Refresh Content | STALE_HIGH_IMPRESSIONS | Medium     | The page may target a niche keyword with stable performance.                                       |
| 15   | Refresh Content | STALE_HIGH_IMPRESSIONS | Medium     | The content could already rank well without requiring significant changes.                         |
| 16   | Refresh Content | STALE_HIGH_IMPRESSIONS | Medium     | The expected improvement may be limited because impressions are relatively low.                    |
| 17   | Refresh Content | STALE_HIGH_IMPRESSIONS | Medium     | Manual review may indicate only small updates are needed instead of a complete refresh.            |
| 18   | Monitor         | NO_ACTION              | High       | The page is relatively recent (20 days), so refreshing it now would likely provide little value.   |
| 19   | Monitor         | NO_ACTION              | High       | Very low impressions suggest there is not enough evidence to prioritize this page yet.             |
| 20   | Monitor         | NO_ACTION              | High       | Although impressions are reasonable, the page is not old enough to meet the baseline rule.         |


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

The weakest picks are the lower-ranked refresh candidates with only slightly more than 500 impressions. Although they satisfy the rule, their expected business impact is smaller than the highest-ranked pages. Some may also be evergreen content that does not require a refresh despite being old.

Leakage Check

The baseline score only uses historical features:

- days_since_last_update
- impressions_90d

No label-derived features (trend_direction, trend_pct) or future information were used. IDs (content_id, client_id) were also excluded from the scoring rule to avoid data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.